# Where to put one new pharmacy? (population-weighted minutes saved)

Notebook `03` gives OSRM driving times from every sampled address to every **active** pharmacy. Notebook `02` gives **`population_weight`** \(w_i\) (people represented per address).

## Objective

If a **new** site \(c\) opens, each address \(i\) gets a new shortest time \(t'_i = \min(t_i, d_{i,c})\) where \(t_i\) is the current best time and \(d_{i,c}\) is drive time to \(c\). **Person–minutes saved** (one-way) are

\[
S(c) = \sum_i w_i \, \max(0,\; t_i - d_{i,c}).
\]

We only consider **OSM settlement points** (`place=village` or `hamlet` only—no `locality`) inside the five study counties, listed in `../data/vt_village_centers.csv`. Regenerate that file with `uv run python scripts/build_vt_village_centers_from_osm.py` (Overpass + county mask). Each place gets a **heuristic score** for sorting tables only: sum of \(w_i \max(0, t_i - T)\) over addresses within `VILLAGE_CATCHMENT_MI` miles (great-circle). **Every** listed settlement is then evaluated with OSRM (one full column of drive times each)—expect a long run; a **progress bar** shows village-level completion. If no local OSRM server responds, the notebook still runs and reports **marginal importance** of *existing* pharmacies (next section).

## Why not “pick the best existing pharmacy column”?

For any column \(k\) already in the network, \(d_{i,k} \ge t_i\), so a duplicate at the same coordinates saves **no** time. A *new* site must be a **different** location (here: OSM village/hamlet coordinates from `vt_village_centers.csv`).

## Marginal value of an *existing* pharmacy

If pharmacy \(j\) **closed**, people reroute to their next-best option. **Person–minutes gained** (lost if it closes) are

\[
L(j) = \sum_i w_i \, \max(0,\; t_i^{(-j)} - t_i)
\]

where \(t_i^{(-j)} = \min_{k \ne j} d_{i,k}\). That uses only the precomputed matrix (no extra OSRM).

In [28]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import requests

DATA_DIR = Path("../data")
ADDRESSES_CSV = DATA_DIR / "addresses_with_population_weights.csv"
DURATION_NPY = DATA_DIR / "osrm_address_pharmacy_duration_sec.npy"
META_JSON = DATA_DIR / "osrm_address_pharmacy_matrix_meta.json"
PHARM_JSON = DATA_DIR / "pharmacies_active.json"
VILLAGE_CENTERS_CSV = DATA_DIR / "vt_village_centers.csv"

MAX_TABLE_COORDS = 100
REQUEST_TIMEOUT_S = 120

# New-site search: OSRM at meta["osrm_base"]; one table column per OSM settlement (many requests).
UNDERSERVED_THRESHOLD_MIN = 15.0
VILLAGE_CATCHMENT_MI = 12.0  # miles: heuristic = sum underserved load for addresses within this radius
OSRM_PROBE_TIMEOUT_S = 3.0

with META_JSON.open() as f:
    meta = json.load(f)
OSRM_BASE = meta["osrm_base"].rstrip("/")

addresses = pd.read_csv(ADDRESSES_CSV)
duration_sec = np.load(DURATION_NPY)
if len(addresses) != duration_sec.shape[0]:
    raise ValueError("Address rows must match duration matrix rows")

pharm_rows = json.loads(PHARM_JSON.read_text(encoding="utf-8"))
by_place = {r["place_id"]: r for r in pharm_rows}
place_order = meta["pharmacy_place_ids"]
if len(place_order) != duration_sec.shape[1]:
    raise ValueError("Meta pharmacy list length must match matrix columns")

w = addresses["population_weight"].to_numpy(dtype=np.float64)
addr_lon = addresses["lon"].to_numpy(dtype=np.float64)
addr_lat = addresses["lat"].to_numpy(dtype=np.float64)

d = np.where(np.isfinite(duration_sec), duration_sec.astype(np.float64), np.inf)
t_nearest_sec = np.min(d, axis=1)
argmin = np.argmin(d, axis=1)
p2 = np.partition(d, 1, axis=1)[:, :2]
t1_sec = p2.min(axis=1)
t2_sec = p2.max(axis=1)
t_min = t_nearest_sec / 60.0

finite = np.isfinite(t_min) & np.isfinite(w) & (w >= 0)
addresses = addresses.loc[finite].reset_index(drop=True)
w = w[finite]
addr_lon = addr_lon[finite]
addr_lat = addr_lat[finite]
d = d[finite]
argmin = argmin[finite]
t1_sec = t1_sec[finite]
t2_sec = t2_sec[finite]
t_min = t_min[finite]

underserved_load = w * np.maximum(0.0, t_min - UNDERSERVED_THRESHOLD_MIN)
print(f"addresses: {len(addresses):,}")
print(f"sum(population_weight) ≈ {w.sum():,.0f}")
print(f"baseline person·min (one-way, nearest): {(w * t_min).sum():,.0f}")

addresses: 86,672
sum(population_weight) ≈ 149,598
baseline person·min (one-way, nearest): 1,746,156


## Marginal loss if each existing pharmacy closed

For pharmacy column \(j\), let \(t^{(-j)}_i = t_{2,i}\) when \(j\) is the current nearest for row \(i\), otherwise \(t^{(-j)}_i = t_i\) (second-smallest on that row handles the former case).

In [29]:
t1_min = t1_sec / 60.0
t2_min = t2_sec / 60.0
n_pharm = d.shape[1]

marginal_person_min = np.empty(n_pharm, dtype=np.float64)
for j in range(n_pharm):
    t_after = np.where(argmin == j, t2_min, t1_min)
    marginal_person_min[j] = float(np.sum(w * np.maximum(0.0, t_after - t1_min)))

marginal_df = pd.DataFrame(
    {
        "col": np.arange(n_pharm),
        "place_id": place_order,
        "person_minutes_lost_if_closed": marginal_person_min,
    }
)
marginal_df["name"] = marginal_df["place_id"].map(lambda pid: by_place.get(pid, {}).get("name", ""))
marginal_df["city"] = marginal_df["place_id"].map(lambda pid: by_place.get(pid, {}).get("city", ""))
marginal_df = marginal_df.sort_values("person_minutes_lost_if_closed", ascending=False)
print("Top 15 pharmacies by population-weighted minutes at risk if that site closes")
marginal_df.head(15)

Top 15 pharmacies by population-weighted minutes at risk if that site closes


,col,place_id,person_minutes_lost_if_closed,name,city
160,160,ChIJccg_vF3XtUwRlZtA9TS8hJs,106202.271962,Kinney Drugs Pharmacy,Barton
77,77,ChIJixYNDKQPtUwRlOqT5UVG0qU,104224.696327,Northfield Pharmacy,Northfield
82,82,ChIJC3UN_tartUwRGxFQalwgsXI,86007.416227,The Health Center,Plainfield
80,80,ChIJTcIqWPxytUwRpSSy_zjDsZ0,80037.724276,Mad River Veterinary Service,Waitsfield
128,128,ChIJ6Y0B8LJHtEwRd9e5GGue6Ns,72753.149484,Genoa Healthcare,St Johnsbury
78,78,ChIJzXRz1PyntUwRL5QvUQy1POw,64387.579258,Kinney Drugs Pharmacy,Montpelier
122,122,ChIJ9WqquVj0tUwR1A5bPgAcSRA,48804.303949,Kinney Drugs Pharmacy,Cambridge
83,83,ChIJq6raH9wGtUwRVo2m5nieoI8,34134.580650,Walgreens Pharmacy,Barre
79,79,ChIJj8r-lFTDVkARMqXUxAcvt1o,15583.641532,Hannaford Pharmacy,Barre
117,117,ChIJqZS_2sqXtUwREYj7NYP2JL8,13855.931210,Lamoille Health Pharmacy,Stowe


## Optimal *new* site among **OSM village / hamlet** coordinates

`vt_village_centers.csv` is built from **OpenStreetMap** (`place=village` or `hamlet` only) clipped to the five-county union—see `scripts/build_vt_village_centers_from_osm.py`. Rows include `name`, `lon`, `lat`, `place`, `osm_type`, `osm_id`.

We sort settlements by **heuristic underserved load** (same formula within `VILLAGE_CATCHMENT_MI` miles) only to make the preview table readable; **OSRM is run for every row**.

For each settlement we request OSRM durations from **every** address to that point, in chunks of 99 addresses + 1 destination (server coordinate limit). If `requests` to `osrm_base` fails the probe below, this section is skipped—start the OSRM server from notebook `03` and re-run.

In [30]:
def haversine_miles_matrix(
    lon1: np.ndarray, lat1: np.ndarray, lon2: np.ndarray, lat2: np.ndarray
) -> np.ndarray:
    """Great-circle miles between two point sets, shape (len(lon1), len(lon2))."""
    r_mi = 3958.7613
    dlon = np.radians(lon1[:, None] - lon2[None, :])
    dlat = np.radians(lat1[:, None] - lat2[None, :])
    p1 = np.radians(lat1)[:, None]
    p2 = np.radians(lat2)[None, :]
    h = np.sin(dlat / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2) ** 2
    return 2 * r_mi * np.arcsin(np.sqrt(np.clip(h, 0.0, 1.0)))


if not VILLAGE_CENTERS_CSV.is_file():
    raise FileNotFoundError(
        f"Missing {VILLAGE_CENTERS_CSV} — run scripts/build_vt_village_centers_from_osm.py"
    )

villages = pd.read_csv(VILLAGE_CENTERS_CSV)
v_lon = villages["lon"].to_numpy(dtype=np.float64)
v_lat = villages["lat"].to_numpy(dtype=np.float64)
dist_mi = haversine_miles_matrix(addr_lon, addr_lat, v_lon, v_lat)
near = dist_mi <= VILLAGE_CATCHMENT_MI
heuristic = near.T @ underserved_load
villages = villages.assign(heuristic_underserved_load=heuristic).sort_values(
    "heuristic_underserved_load", ascending=False
)

cand = villages.reset_index(drop=True)
print(
    f"{len(cand)} OSM settlements — all will be evaluated with OSRM (≈{len(cand) * int(np.ceil(len(addr_lon) / (MAX_TABLE_COORDS - 1))):,} table requests)"
)
preview_cols = [c for c in ("name", "place", "lon", "lat", "heuristic_underserved_load", "osm_type", "osm_id", "source") if c in cand.columns]
cand[preview_cols].head(25)

237 OSM settlements — all will be evaluated with OSRM (≈207,612 table requests)


,name,place,lon,lat,heuristic_underserved_load,osm_type,osm_id,source
0,Tolman Corner,hamlet,-72.306755,44.567145,120937.526566,node,158859015,osm_overpass
1,Greensboro,village,-72.295681,44.576004,120685.753355,node,158863131,osm_overpass
2,Hardwick Street,hamlet,-72.299549,44.549775,119596.625071,node,158799294,osm_overpass
3,Hardwick Center,hamlet,-72.336799,44.529498,118903.076206,node,158843116,osm_overpass
4,Campbells Corner,hamlet,-72.331495,44.591719,118630.311826,node,158853498,osm_overpass
5,East Greensboro,hamlet,-72.250659,44.571442,117639.833188,node,158843152,osm_overpass
6,Greensboro Bend,hamlet,-72.264826,44.548942,117071.904413,node,158839100,osm_overpass
7,East Craftsbury,hamlet,-72.342806,44.641510,116846.716220,node,158899116,osm_overpass
8,East Hardwick,hamlet,-72.308716,44.521721,116517.269253,node,158825132,osm_overpass
9,Craftsbury,village,-72.373296,44.636106,115718.882379,node,158858815,osm_overpass


In [40]:
def osrm_durations_to_one_dest(
    osrm_base: str,
    src_lon: np.ndarray,
    src_lat: np.ndarray,
    dest_lon: float,
    dest_lat: float,
) -> np.ndarray:
    """Driving duration in seconds from each source coordinate to one destination."""
    n = len(src_lon)
    out = np.full(n, np.nan, dtype=np.float64)
    chunk = MAX_TABLE_COORDS - 1
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        alon = src_lon[start:end]
        alat = src_lat[start:end]
        na = end - start
        coords = ";".join([f"{lo},{la}" for lo, la in zip(alon, alat)] + [f"{dest_lon},{dest_lat}"])
        sources = ";".join(str(i) for i in range(na))
        destinations = str(na)
        url = f"{osrm_base}/table/v1/driving/{coords}"
        r = requests.get(
            url,
            params={
                "sources": sources,
                "destinations": destinations,
                "annotations": "duration",
            },
            timeout=REQUEST_TIMEOUT_S,
        )
        r.raise_for_status()
        data = r.json()
        if data.get("code") != "Ok":
            raise RuntimeError(data)
        for i, row in enumerate(data["durations"]):
            v = row[0]
            out[start + i] = np.nan if v is None else float(v)
    return out


def person_minutes_saved_new_site(t_current_min: np.ndarray, w_row: np.ndarray, d_new_sec: np.ndarray) -> float:
    d_new_min = np.where(np.isfinite(d_new_sec), d_new_sec / 60.0, np.inf)
    return float(np.sum(w_row * np.maximum(0.0, t_current_min - d_new_min)))


osrm_ok = False
try:
    pr = requests.get(
        f"{OSRM_BASE}/nearest/v1/driving/-72.58,44.26",
        timeout=OSRM_PROBE_TIMEOUT_S,
    )
    osrm_ok = pr.ok
except (requests.RequestException, OSError):
    osrm_ok = False

from rich.progress import BarColumn, Progress, TaskProgressColumn, TextColumn, TimeElapsedColumn, TimeRemainingColumn

placement_rows: list[dict] = []
if not osrm_ok:
    print("OSRM not reachable; skipping new-site evaluation (marginal table above is still valid).")
    placement_df = pd.DataFrame()
else:
    n_c = len(cand)
    per_village_reqs = int(np.ceil(len(addr_lon) / (MAX_TABLE_COORDS - 1)))
    print(
        f"OSRM at {OSRM_BASE} — {n_c} settlements × ~{per_village_reqs:,} table requests each"
    )
    with Progress(
        TextColumn("[bold]OSRM[/]"),
        BarColumn(),
        TaskProgressColumn(),
        TimeElapsedColumn(),
        TimeRemainingColumn(),
        TextColumn("{task.description}"),
    ) as progress:
        task = progress.add_task("", total=n_c)
        for _, row in cand.iterrows():
            label = str(row["name"])[:44]
            progress.update(task, description=label)
            dlon, dlat = float(row["lon"]), float(row["lat"])
            d_sec = osrm_durations_to_one_dest(OSRM_BASE, addr_lon, addr_lat, dlon, dlat)
            saved = person_minutes_saved_new_site(t_min, w, d_sec)
            oid = row["osm_id"] if "osm_id" in row and pd.notna(row["osm_id"]) else None
            placement_rows.append(
                {
                    "village": row["name"],
                    "place": row["place"] if "place" in row and pd.notna(row["place"]) else "",
                    "lon": dlon,
                    "lat": dlat,
                    "osm_id": oid,
                    "heuristic_underserved_load": row["heuristic_underserved_load"],
                    "person_minutes_saved_one_way": saved,
                }
            )
            progress.advance(task)
    placement_df = pd.DataFrame(placement_rows).sort_values(
        "person_minutes_saved_one_way", ascending=False
    )
    print("Top settlements by person·minutes saved (one-way)")
    placement_df.head(15)

Output()

OSRM at http://127.0.0.1:8008 — 237 settlements × ~876 table requests each


KeyboardInterrupt: 

In [45]:
placement_df.to_csv("../data/placement_hypothetical_sites.csv", index=False)

### Interpretation

- **Marginal** table: “insurance” value of keeping each current pharmacy open (by rerouting cost).
- **Placement** table: ranks **every** OSM settlement in `vt_village_centers.csv` by true person·minutes saved. After OSRM finishes, results are written to `../data/placement_hypothetical_sites.csv`. Map those points with `uv run python scripts/build_placement_sites_map.py` → `placement_hypothetical_sites_map.html`. Regenerate settlements with `scripts/build_vt_village_centers_from_osm.py`. Tune `VILLAGE_CATCHMENT_MI` only for the heuristic sort / preview.
- Round-trip savings are simply \(2 \times S(c)\) if you assume symmetric return travel and the same destination.